###**Installing and Importing the necessary libraries/packages**

In [ ]:
# Install specific versions of required libraries for the Q&A system
!pip install -q openai==1.23.2 \
                tiktoken==0.6.0 \
                pypdf==4.0.1 \
                langchain==0.1.1 \
                langchain-community==0.0.13 \
                chromadb==0.4.22 \
                sentence-transformers==2.3.1
!pip install azure-ai-textanalytics
!pip install tiktoken
!pip install gradio
!pip install camelot-py

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 5.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 311.2/311.2 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 30.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 284.0/284.0 kB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 802.4/802.4 kB 42.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 47.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 509.0/509.0 kB 30.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.8/132.8 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 70.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.8/273.8 kB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.7/94.7 kB 9.6 MB/s eta

In [ ]:
#mounting the google drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
#importing the necessary libraries
from collections import Counter
import matplotlib.pyplot as plt
import json
import tiktoken
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_community.vectorstores import Chroma
import openai
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.embeddings.sentence_transformer import SentenceTransformerEmbeddings
from langchain.schema import Document # import Document class
import gradio as gr
import pickle
from openai import AzureOpenAI
from google.colab import userdata
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer, util
from langchain.docstore.document import Document
from langchain.prompts import PromptTemplate
from langchain.chains import RetrievalQA
from google.colab import drive
from langchain.embeddings import AzureOpenAIEmbeddings
from langchain.vectorstores import Chroma
import os
import camelot


/usr/local/lib/python3.10/dist-packages/pypdf/_crypt_providers/_cryptography.py:32: CryptographyDeprecationWarning: ARC4 has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.ARC4 and will be removed from this module in 48.0.0.
  from cryptography.hazmat.primitives.ciphers.algorithms import AES, ARC4


###**Building the client for the LLM model**





In [ ]:
# building the client for the llm model
azure_api_key = userdata.get('w5d5')
client = AzureOpenAI(
    azure_endpoint = "https://resrcgrp1.openai.azure.com/",
    api_key=azure_api_key,
    api_version="2024-05-01-preview")
model_name = "gpt-35-turbo"

In [ ]:
azure_api_key_eval = userdata.get('eval')
client_eval = AzureOpenAI(
    azure_endpoint = "https://resrcgrp1.openai.azure.com/",
    api_key=azure_api_key_eval,
    api_version="2024-05-01-preview")
model_name_eval = "gpt-4"

###**Initializing the embedding model**

In [ ]:
# embedding model for initialization and adding documents
key="22f8a591095d4d0084a86db66cd8b3bc"  # Replace with your actual key
embedding_model = AzureOpenAIEmbeddings(
    model="text-embedding-3-large",
    azure_endpoint="https://resrcgrp1.openai.azure.com/openai/deployments/text-embedding-3-large/embeddings?api-version=2023-05-15",
    api_key=key)


/usr/local/lib/python3.10/dist-packages/langchain_core/_api/deprecation.py:117: LangChainDeprecationWarning: The class `langchain_community.embeddings.azure_openai.AzureOpenAIEmbeddings` was deprecated in langchain-community 0.1.0 and will be removed in 0.2.0. An updated version of the class exists in the langchain-openai package and should be used instead. To use it run `pip install -U langchain-openai` and import as `from langchain_openai import AzureOpenAIEmbeddings`.
  warn_deprecated(


###**Declaring the persisted vectorstore**

In [ ]:
persist_directory = '/content/drive/MyDrive/Report' #the path to the persisted vectorstore

v_alphabet = Chroma(persist_directory=persist_directory, embedding_function=embedding_model)# Load the persisted vectorstore

r_alphabet = v_alphabet.as_retriever( # The retriever for the persisted vector database
    search_type='similarity',
    search_kwargs={'k': 5}
)
user_input = "What was the annual revenue of the company in 2022?"

relevant_document_chunks = r_alphabet.get_relevant_documents(user_input)

for document in relevant_document_chunks:
    print(document.page_content.replace("\t", " "))
    break

Year Ended December 31,
2020 2021 2022
Revenues:
Google Services $ 168,635 $ 237,529 $ 253,528
Google Cloud 13,059 19,206 26,280
Other Bets 657 753 1,068
Hedging gains (losses) 176 149 1,960
Total revenues $ 182,527 $ 257,637 $ 282,836
Operating income (loss):
Google Services $ 54,606 $ 91,855 $ 86,572
Google Cloud (5,607 ) (3,099 ) (2,968 )
Other Bets (4,476 ) (5,281 ) (6,083 )
Corporate costs, unallocated (3,299 ) (4,761 ) (2,679 )
Total income from operations $ 41,224 $ 78,714 $ 74,842


In [ ]:
# Initialize an empty Chroma vectorstore without any placeholder document
vectorstore = Chroma(
    collection_name='financial_reports',
    embedding_function=embedding_model  # Optional: Specify a directory to persist the database
)

####**Prompt Templates**

In [ ]:
# Instructions for the LLM on how to behave as a specialized financial assistant.
qna_system_message = """
You are a specialized assistant for a financial services firm, designed to answer user queries based on annual reports.
Your goal is to analyze the provided context from financial documents and answer user questions by following a step-by-step process.
Along with the question the prior conversations will also be provided to guide the answer.

###Follow these steps while answering the query:
1. **Understand the question**: Carefully read and understand the question being asked.
2. **Analyze the context**: Break down the provided context into relevant parts that answer the question.
3. **Step-by-step reasoning**: Use a chain of thought reasoning process to guide your answer. Walk through the steps needed to derive the answer.
4. **Formulate the answer**: Ensure the answer is accurate, concise, and based solely on the provided context.
5. **State if unknown**: If the answer cannot be found in the provided context, simply respond with "I don't know."
This context will begin with the token: ###Context.
The context contains references to specific portions of a document relevant to the user query.

Previous interactions will begin with the token: ###Conversation.
The conversation contains prior user questions and assistant responses that may help in answering the current query.

User questions will begin with the token: ###Question.

Your goal is to provide accurate, concise, and relevant answers.
Avoid mentioning unnecesaary information which is not related to the question while giving the response.

If the answer is not found in the context, respond "I don't know".
Always strive to maintain professionalism and clarity in your responses.
"""

# Template for passing context and the user’s question to the LLM.
qna_user_message_template = """
###Context
The following documents contain information pertinent to the question posed:
{context}

###Question
Please answer the question based on the provided context:
{question}

###Chain of Thought
1. **Understanding the Question**: Clearly identify what the user is asking.
2. **Identifying Key Information in Context**: Highlight the most relevant parts of the provided context that directly answer the question.
3. **Formulate the Answer**: Based on the key information identified, provide the correct answer.
4. **Conclusion**: Summarize your final answer concisely.
Only give the final answer and show the steps only if the user asks for it.

###Answer
"""

context_for_query = ""

####**Loading the Data**

In [ ]:
# function to extract the data from the report
def process_pdf_with_tables(pdf_file):
    text_chunks = []
    table_chunks = []

    try:
        # Load PDF content
        loader = PyPDFLoader(pdf_file.name)
        text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
            encoding_name='cl100k_base', chunk_size=512, chunk_overlap=20
        )
        text_chunks = loader.load_and_split(text_splitter)
        print(f"Number of text chunks extracted: {len(text_chunks)}")

        # Extract tabular data from PDF using Camelot
        tables = camelot.read_pdf(pdf_file.name, pages='all')

        for i, table in enumerate(tables):
            df_table = table.df
            table_text = df_table.to_string(index=False, header=False)  # Convert table to text format
            doc = Document(page_content=table_text)
            table_chunks.append(doc)
        print(f"Number of tables extracted: {len(table_chunks)}")

        return text_chunks, table_chunks

    except Exception as e:
        print(f"Error processing PDF: {e}")
        return text_chunks, table_chunks


###**Anwering the question**

In [ ]:
# function to answer user questions based on the content of a PDF file, including both text and tables.
vectorstore= None
def answer_question_with_tables(pdf_file, user_input, model_choice):
    global vectorstore  # Use global to modify the existing vectorstore variable

    try:
        # Clear the old vectorstore or create a new instance
        vectorstore = Chroma(
            collection_name='financial_reports',
            embedding_function=embedding_model,
            persist_directory='./chroma_db'  # Specify a directory to persist the database
        )

        print("Loading PDF content...")
        # Load PDF content and extract both text and tables
        text_chunks, table_chunks = process_pdf_with_tables(pdf_file)

        # Add text chunks and table chunks to the vectorstore
        if text_chunks:
            vectorstore.add_documents(text_chunks)
        if table_chunks:
            vectorstore.add_documents(table_chunks)


        print("Documents added to vectorstore. Retrieving relevant chunks...")

        # Retrieve relevant documents based on the user query
        relevant_document_chunks = vectorstore.similarity_search(user_input)
        if not relevant_document_chunks:
            print("I don't know.")
            return "I don't know.", ""

        context_list = [d.page_content for d in relevant_document_chunks]
        context_for_query = ". ".join(context_list)

        # Choose model for generating response
        if model_choice == "GPT-3.5 Turbo":
            client_gen = client
            model_gen_name = model_name
        else:
            client_gen = client_eval
            model_gen_name = model_name_eval

        # Prepare the prompt for generating the answer
        prompt = [
            {'role': 'system', 'content': qna_system_message},
            {'role': 'user', 'content': qna_user_message_template.format(
                context=context_for_query,
                question=user_input
            )}
        ]

        # Generate the answer
        response = client_gen.chat.completions.create(
            model=model_gen_name,
            messages=prompt,
            temperature=0.1,
            max_tokens=500,
            top_p=0.9,
            frequency_penalty=0,
            presence_penalty=0
        )
        prediction = response.choices[0].message.content.strip()

        return prediction, context_for_query

    except Exception as e:
        return f"Error in answer_question: {e}", ""


###**Evaluating the response generated**

In [ ]:
# function to evaluate the quality of the AI's response using a secondary LLM.
def evaluate_response(user_input, response_text, context_for_query, model_choice):
    try:
        # The prompt to be followed for evaluation
        evaluation_prompt = [
            {'role': 'system', 'content': """
        You are an evaluator of the quality of answers provided by an AI assistant.
        Your job is to determine how well the AI response accurately answers the question, and whether it adheres to the context.
        Your evaluation should consider the accuracy, relevance, clarity, and completeness of the response.

        You will receive a user question, an AI response, and the context used by the AI to generate the response.

        Please provide a score between 0 and 100, where:
        - 100 indicates a perfect answer, relevant, and comprehensive.
        - 0 indicates a completely irrelevant or nonsensical response.

        Additionally, provide a brief explanation of your score, justifying your evaluation.
        """},
        {'role': 'user', 'content': f"""
        ### User Question:
        {user_input}

        ### AI Response:
        {response_text}

        ### Context:
        {context_for_query}

        Please provide your evaluation score and explanation based on these criteria: relevance, clarity, and chain of thought reasoning.
        """}
        ]

        # Choose evaluation model
        if model_choice == "GPT-3.5 Turbo":
            eval_model = model_name_eval  # GPT-4 for evaluating
            eval_client = client_eval
        else:
            eval_model = model_name  # GPT-3.5 Turbo for evaluating
            eval_client = client

        # Get evaluation
        response_eval = eval_client.chat.completions.create(
            model=eval_model,
            messages=evaluation_prompt,
            temperature=0,
            max_tokens=300,          # Limits the evaluation length
            top_p=1.0,               # No nucleus sampling restriction
            frequency_penalty=0.5,   # Prevents repetitive phrases
            presence_penalty=0.2
        )

        evaluation_result = response_eval.choices[0].message.content.strip()
        print(f"Evaluation result: {evaluation_result}")
        return evaluation_result

    except Exception as e:
        print(f"Error in evaluate_response: {e}")
        return f"Error in evaluate_response: {e}"


###**Resetting the vectorstore for next report**

In [ ]:
# Global variable to track the current PDF file
current_pdf_file = None
conversation_history = []
vectorstore = None  # Initialize the vector store globally

def reset_vector_store():
    """
    Function to clear the vector store or reset it for new context.
    This function reinitializes the vectorstore to ensure it starts fresh.
    """
    global vectorstore  # Use the global variable
    vectorstore = Chroma(
        collection_name='financial_reports',
        embedding_function=embedding_model,
        persist_directory='./chroma_db'  # Specify a directory to persist the database
    )
    print("Vector store has been reset.")


###**Setting up the system**

In [ ]:
def qa_system_with_memory(pdf_file, question, model_choice, use_persisted_db):
    global conversation_history, current_pdf_file  # Reference the global conversation history and current PDF

    # Check if the PDF has changed
    if pdf_file and (current_pdf_file is None or pdf_file.name != current_pdf_file.name):
        # If a new PDF is uploaded, reset the conversation history
        conversation_history = []
        current_pdf_file = pdf_file  # Update the current PDF to the new one

        # Reset the vector store
        reset_vector_store()

    context_for_query = ""  # Initialize context
    evaluation = ""  # Initialize evaluation

    # Check if persisted vectorstore or a new PDF is to be used
    if use_persisted_db:
        # Use the persisted vectorstore
        retriever = r_alphabet  # Assuming `r_alphabet` is properly initialized
        g="\n".join([f"User: {q}\nAI: {r}" for q, r in conversation_history])
        relevant_document_chunks = retriever.get_relevant_documents(question+g)

        if not relevant_document_chunks:
            return "No relevant documents found in the persisted database.", conversation_history, ""

        context_list = [d.page_content for d in relevant_document_chunks]
        context_for_query = ". ".join(context_list)
    else:
        # Process a new document using the updated function for tables
        g="\n".join([f"User: {q}\nAI: {r}" for q, r in conversation_history])
        response, context_for_query = answer_question_with_tables(pdf_file, question+g, model_choice)
        if not context_for_query:  # If context is empty or there was an error
            return response, conversation_history, ""  # Return only the response and no evaluation

    # Choose model for generating response
    if model_choice == "GPT-3.5 Turbo":
        client_gen = client
        model_gen_name = model_name
    else:
        client_gen = client_eval
        model_gen_name = model_name_eval

    # Add previous conversation to the context for memory
    conversation_text = "\n".join([f"User: {q}\nAI: {r}" for q, r in conversation_history])

    # Include conversation history in the system prompt for memory
    system_prompt = qna_system_message + f"\n###Conversation\n{conversation_text}"

    # Prepare the prompt for generating the answer
    prompt = [
        {'role': 'system', 'content': system_prompt},
        {'role': 'user', 'content': qna_user_message_template.format(
            context=context_for_query,
            question=question
        )}
    ]

    # Generate the answer
    response = client_gen.chat.completions.create(
        model=model_gen_name,
        messages=prompt,
        temperature=0.1,
        max_tokens=500,
        top_p=0.9,
        frequency_penalty=0,
        presence_penalty=0
    )
    prediction = response.choices[0].message.content.strip()

    # Save the new question and response in the conversation history
    conversation_history.append((question, prediction))

    # Evaluate only the current response, not the entire conversation history
    evaluation = evaluate_response(question, prediction, context_for_query, model_choice)

    # Return the response, updated conversation history, and evaluation score
    return prediction, conversation_history, evaluation


###**Setting up the Gradio Interface**

In [ ]:
def clear_interface(): #clearing
    """
    Function to clear all inputs and outputs, and reset the vector store.
    """
    global conversation_history, vectorstore
    conversation_history = []  # Clear conversation history
    vectorstore = None  # Reset the vector store
    reset_vector_store()  # Call to ensure vector store is reset
    return None, [], "", None, ""  # Clear values for outputs

# Gradio Interface with layout adjustments
with gr.Blocks() as iface:
    gr.Markdown("# Financial Report Q&A System")
    gr.Markdown("Upload a financial report or use the reports available in the persisted database to ask questions and see previous interactions.")

    with gr.Row():
        # Input Column (1/4 of the screen)
        with gr.Column(scale=1):  # Scale of 1 means this will take 1/4 of the width
            pdf_file_input = gr.File(label="Upload PDF Report")  # File input for PDF
            question_input = gr.Textbox(label="Your Question", placeholder="Type your question here...")  # Textbox for user's question
            model_choice = gr.Dropdown(choices=["GPT-3.5 Turbo", "GPT-4"], label="Choose Model")  # Model choice
            use_persisted_db = gr.Checkbox(label="Use Persisted VectorDatabase - Alphabet Inc.", value=True)  # Checkbox to use the persisted DB

            # Clear and Submit buttons
            clear_button = gr.Button("Clear")  # Clear button
            submit_button = gr.Button("Submit")  # Submit button

        # Output Column (3/4 of the screen)
        with gr.Column(scale=3):  # Scale of 3 means this will take 3/4 of the width
            response_output = gr.Textbox(label="Response", interactive=False)  # Show the response
            chatbot_output = gr.Chatbot(label="Conversation History")  # Chat-style interface for showing history
            evaluation_output = gr.Textbox(label="Evaluation Score and Explanation", interactive=False)  # Show evaluation for current response

    # Define button actions
    submit_button.click(qa_system_with_memory,
                        inputs=[pdf_file_input, question_input, model_choice, use_persisted_db],
                        outputs=[response_output, chatbot_output, evaluation_output]
    ).then(
    lambda: "",  # Clear the question input after response generation
    inputs=None,
    outputs=question_input)  # Reset the question input box to blank


    # Clear action
    clear_button.click(clear_interface, outputs=[response_output, chatbot_output, evaluation_output, pdf_file_input, question_input])

# Launch the Gradio interface
iface.launch()

/usr/local/lib/python3.10/dist-packages/gradio/components/chatbot.py:222: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  warnings.warn(


Running Gradio in a Colab notebook requires sharing enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://cc5a5f17f25d370a9b.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
